# Imports

In [98]:
import pandas as pd
import os
from pathlib import Path
import re

In [99]:
if 'notebooks' in os.getcwd():
    print("Still in notebooks/ directory. Changing to root directory..")
    os.chdir('..')
print(f"Already in root directory: {os.getcwd()}")

Already in root directory: d:\data-sci-projects\aer-data-extraction


# Processed descriptor and cost metrics

In [100]:
os.listdir('data/processed')

['rin_maintenance_cost_metrics.csv',
 'rin_maintenance_descriptor_metrics.csv',
 'rin_maintenance_run_report.csv']

In [101]:
desc = pd.read_csv('data/processed/rin_maintenance_descriptor_metrics.csv')
cost = pd.read_csv('data/processed/rin_maintenance_cost_metrics.csv')

In [102]:
desc

,reporting_period,maintenance_activity,maintenance_asset_category,measure_asset_quantity,source_unit,asset_quantity_at_year_end,quantity_inspected_maintained,average_age_of_asset_group,inspection_cycle_years,maintenance_cycle_years,source_workbook,source_sheet,source_row
0,2019-20,Transmission lines maintenance,Transmission towers,Number of towers,0's,13204.00,17605.333333,49.797561,1.0,3.0,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,12
1,2019-20,Transmission lines maintenance,Transmission tower support structures,Number of towers,0's,69527.00,46351.333333,49.039255,3.0,3.0,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,13
2,2019-20,Transmission lines maintenance,Conductors,Route length,km,5086.80,1695.600000,45.342263,3.0,0.0,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,14
3,2019-20,Transmission lines maintenance,Transmission cables,Route length,km,8.54,17.080000,26.126850,1.0,1.0,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,15
4,2019-20,Substations equipment & property maintenance,Substation switchbays (incl. Reactive plant),Number of switchbays,0's,1133.00,2340.000000,22.911932,1.0,6.0,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...
280,2023-24,Substations equipment & property maintenance,Substation property,Number of substation properties maintained,number,96.00,95.000000,39.900000,0.5,0.3,Transgrid 2023-24 - Category Analysis - RIN Re...,2.8 Maintenance,19
281,2023-24,SCADA & network control maintenance,SCADA & network control maintenance,Number of main control assets,0's,2046.00,196.000000,12.600000,NaN,NaN,Transgrid 2023-24 - Category Analysis - RIN Re...,2.8 Maintenance,20
282,2023-24,Protection systems maintenance,Protection systems maintenance,Number of main protection assets,0's,3286.00,893.000000,15.600000,NaN,6.5,Transgrid 2023-24 - Category Analysis - RIN Re...,2.8 Maintenance,21
283,2023-24,Other maintenance activity,Metering,Number of main metering assets,0's,838.00,639.000000,10.700000,2.5,5.0,Transgrid 2023-24 - Category Analysis - RIN Re...,2.8 Maintenance,22


In [103]:
cost

,reporting_period,maintenance_activity,maintenance_asset_subcategory,source_currency_unit,routine_maintenance_expenditure,non_routine_maintenance_expenditure,source_workbook,source_sheet,source_row
0,2019-20,Transmission lines maintenance,Transmission towers,$0's,1.090083e+06,8.798923e+05,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,50
1,2019-20,Transmission lines maintenance,Transmission tower support structures,$0's,1.711115e+05,1.089070e+05,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,51
2,2019-20,Transmission lines maintenance,Conductors,$0's,9.506952e+05,4.673437e+05,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,52
3,2019-20,Transmission lines maintenance,Transmission cables,$0's,1.230585e+05,2.483836e+04,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,53
4,2019-20,Substations equipment & property maintenance,Substation switchbays (incl. Reactive plant),$0's,3.323640e+06,1.442763e+06,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,54
...,...,...,...,...,...,...,...,...,...
284,2023-24,Substations equipment & property maintenance,Substation property,$,6.745046e+06,4.178675e+06,Transgrid 2023-24 - Category Analysis - RIN Re...,2.8 Maintenance,57
285,2023-24,SCADA & network control maintenance,SCADA & network control maintenance,$,2.443070e+05,7.348910e+05,Transgrid 2023-24 - Category Analysis - RIN Re...,2.8 Maintenance,58
286,2023-24,Protection systems maintenance,Protection systems maintenance,$,9.816660e+05,8.727580e+05,Transgrid 2023-24 - Category Analysis - RIN Re...,2.8 Maintenance,59
287,2023-24,Other maintenance activity,Metering,$,5.329630e+05,2.245490e+05,Transgrid 2023-24 - Category Analysis - RIN Re...,2.8 Maintenance,60


# Inspect nulls

These may not be genuine nulls because of say excel merges.
For descriptor table we have
```python
desc_categs = ['maintenance_activity', 'maintenance_asset_category']
desc_units = ['measure_asset_quantity', 'source_unit']
desc_nums = ['asset_quantity_at_year_end',
            'quantity_inspected_maintained',
            'average_age_of_asset_group',
            'inspection_cycle_years',
            'maintenance_cycle_years']
```
Possible cases: (not each null/non null = **at least** one row)
1. `desc_categs` non null, `desc_nums` null
2. `desc_categs` non null, `desc_nums` non null
3. `desc_categs` null, `desc_nums` null
4. `desc_categs` null, `desc_nums` non null

Case 3 and 4 are of interest.

## Null category, valid numeric observation

In [8]:
# desc_categs = ['maintenance_activity', 'maintenance_asset_category']
# desc_units = ['measure_asset_quantity', 'source_unit']
# desc_nums = ['asset_quantity_at_year_end',
#             'quantity_inspected_maintained',
#             'average_age_of_asset_group',
#             'inspection_cycle_years',
#             'maintenance_cycle_years']

# # obtain at least one non-null numeric observations with at least one null category
# null_categs_locator = desc[desc_categs].isna().any(axis=1)
# non_null_num_locator = desc[desc_nums].notna().any(axis=1)

# valid_num_null_categs = desc[null_categs_locator & non_null_num_locator]
# valid_num_null_categs_wkbk_list = valid_num_null_categs.source_workbook.unique().tolist()


# for wkbk in valid_num_null_categs_wkbk_list:
#     print('-'*10 + f'{wkbk}' + '-'*10)
#     display(desc.query(f'source_workbook == "{wkbk}"'))

In [9]:
# cost_categs = ['maintenance_activity', 'maintenance_asset_subcategory']
# cost_units = ['source_currency_unit']
# cost_nums = ['routine_maintenance_expenditure',
#             'non_routine_maintenance_expenditure']

# # obtain at least one non-null numeric observations with at least one null category
# null_categs_locator = cost[cost_categs].isna().any(axis=1)
# non_null_num_locator = cost[cost_nums].notna().any(axis=1)

# valid_num_null_categs = cost[null_categs_locator & non_null_num_locator]
# valid_num_null_categs_wkbk_list = valid_num_null_categs.source_workbook.unique().tolist()


# for wkbk in valid_num_null_categs_wkbk_list:
#     print('-'*10 + f'{wkbk}' + '-'*10)
#     display(cost.query(f'source_workbook == "{wkbk}"'))

# Stage 2A

In [104]:
from src.rin_maintenance_standardizer import enrich_rin_maintenance

In [105]:
run_report = pd.read_csv('data/processed/rin_maintenance_run_report.csv')
manifest = pd.read_csv('data/rin_manifest.csv')

In [106]:
stage_2a = enrich_rin_maintenance(
    descriptor_metrics=desc,
    cost_metrics=cost,
    run_report=run_report,
    manifest=manifest
)

## Enriched cost metrics

In [107]:
display(stage_2a.cost_metrics.head())
display(stage_2a.cost_metrics.columns)

,reporting_period,maintenance_activity,maintenance_asset_subcategory,source_currency_unit,routine_maintenance_expenditure,non_routine_maintenance_expenditure,source_workbook,source_sheet,source_row,business,landing_page_url,source_page_url,metadata_match_status,maintenance_activity_resolved,activity_resolution_status,activity_anchor_source_row,row_classification
0,2019-20,Transmission lines maintenance,Transmission towers,$0's,1.090083e+06,8.798923e+05,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,50,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match,Transmission lines maintenance,submitted,50,meaningful
1,2019-20,Transmission lines maintenance,Transmission tower support structures,$0's,1.711115e+05,1.089070e+05,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,51,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match,Transmission lines maintenance,submitted,51,meaningful
2,2019-20,Transmission lines maintenance,Conductors,$0's,9.506952e+05,4.673437e+05,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,52,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match,Transmission lines maintenance,submitted,52,meaningful
3,2019-20,Transmission lines maintenance,Transmission cables,$0's,1.230585e+05,2.483836e+04,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,53,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match,Transmission lines maintenance,submitted,53,meaningful
4,2019-20,Substations equipment & property maintenance,Substation switchbays (incl. Reactive plant),$0's,3.323640e+06,1.442763e+06,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,54,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match,Substations equipment & property maintenance,submitted,54,meaningful


Index(['reporting_period', 'maintenance_activity',
       'maintenance_asset_subcategory', 'source_currency_unit',
       'routine_maintenance_expenditure',
       'non_routine_maintenance_expenditure', 'source_workbook',
       'source_sheet', 'source_row', 'business', 'landing_page_url',
       'source_page_url', 'metadata_match_status',
       'maintenance_activity_resolved', 'activity_resolution_status',
       'activity_anchor_source_row', 'row_classification'],
      dtype='str')

In [108]:
stage_2a.cost_metrics.activity_resolution_status.unique()

<StringArray>
['submitted', 'continued_group_label']
Length: 2, dtype: str

In [110]:
stage_2a.cost_metrics.metadata_match_status.unique()

<StringArray>
['validated_manifest_match']
Length: 1, dtype: str

### Verify: all 24 workbooks reconcile to the expected businesses

In [111]:
stage_2a.cost_metrics[['source_workbook', 'business']].drop_duplicates()

,source_workbook,business
0,AusNet (T) 2019-20 - Category Analysis - RIN R...,AusNet Transmission
10,AusNet (T) 2020-21 - Category Analysis - RIN R...,AusNet Transmission
20,AusNet (T) 2021-22 - Category Analysis - RIN R...,AusNet Transmission
30,Ausnet Services (ET) 2022-23 - Category Analys...,AusNet Transmission
40,Ausnet Services (ET) 2023-24 - Category Analys...,AusNet Transmission
50,Copy of D14-00150087(V02) - transgrid (t) 2013...,Transgrid
62,Copy of D16 147530(V3) ElectraNet 2015-16 - C...,ElectraNet
72,Copy of D17 153782 ElectraNet 2016-17 - Categ...,ElectraNet
82,DORIS - D18-158142 Transgrid 2017-18 - Categor...,Transgrid
93,DORIS - D18-158267 ElectraNet 2017-18 - Catego...,ElectraNet


### Verify: there are no unexpected unresolved rows;

In [112]:
not(
    'unresolved' in stage_2a.cost_metrics['row_classification'].unique()
)

True

### Verify: continued rows inherit the correct parent activity

In [113]:
stage_2a.cost_metrics[['business', 'reporting_period', 'maintenance_activity',
       'maintenance_asset_subcategory', 'maintenance_activity_resolved',
       'activity_resolution_status', 'activity_anchor_source_row',
       'row_classification']].query('activity_resolution_status != "submitted" ')

,business,reporting_period,maintenance_activity,maintenance_asset_subcategory,maintenance_activity_resolved,activity_resolution_status,activity_anchor_source_row,row_classification
60,Transgrid,2013-14,NaN,METERING SYSTEMS,Other maintenance activity,continued_group_label,21,meaningful
61,Transgrid,2013-14,NaN,TRANSMISSION LINES MAINTENANCE ACCESS TRACKS,Other maintenance activity,continued_group_label,21,meaningful
92,Transgrid,2017-18,NaN,Communications,Other maintenance activity,continued_group_label,59,meaningful
103,ElectraNet,2017-18,NaN,Row Maintenance,Other maintenance activity,continued_group_label,59,meaningful
174,Powerlink,2019-20,NaN,CORRIDOR MAINTENANCE (NON VEG),Other maintenance activity,continued_group_label,59,meaningful
185,Powerlink,2020-21,NaN,Corridor maintenance (Non vegetation),Other maintenance activity,continued_group_label,59,meaningful
196,Powerlink,2021-22,NaN,Corridor maintenance (non vegetation),Other maintenance activity,continued_group_label,59,meaningful
207,Powerlink,2022-23,NaN,Corridor maintenance (non veg),Other maintenance activity,continued_group_label,60,meaningful
218,Powerlink,2023-24,NaN,Corridor Maintenance (Non-Veg),Other maintenance activity,continued_group_label,60,meaningful
229,Transgrid,2018-19,NaN,Communications,Other maintenance activity,continued_group_label,59,meaningful


### Verify: the group_header_only row is sensible

In [114]:
# parent heading 'Other maintenance activity' has no child or any numerical metrics
stage_2a.cost_metrics[
    stage_2a.cost_metrics.row_classification == 'group_header_only'
]

,reporting_period,maintenance_activity,maintenance_asset_subcategory,source_currency_unit,routine_maintenance_expenditure,non_routine_maintenance_expenditure,source_workbook,source_sheet,source_row,business,landing_page_url,source_page_url,metadata_match_status,maintenance_activity_resolved,activity_resolution_status,activity_anchor_source_row,row_classification
102,2017-18,Other maintenance activity,NaN,$0's,NaN,NaN,DORIS - D18-158267 ElectraNet 2017-18 - Catego...,2.8 Maintenance,59,ElectraNet,https://www.aer.gov.au/documents/electranet-20...,https://www.aer.gov.au/authors/electranet,validated_manifest_match,Other maintenance activity,submitted,59,group_header_only


## Enriched descriptor metrics

In [115]:
display(stage_2a.descriptor_metrics.head())
display(stage_2a.descriptor_metrics.columns)

,reporting_period,maintenance_activity,maintenance_asset_category,measure_asset_quantity,source_unit,asset_quantity_at_year_end,quantity_inspected_maintained,average_age_of_asset_group,inspection_cycle_years,maintenance_cycle_years,...,source_sheet,source_row,business,landing_page_url,source_page_url,metadata_match_status,maintenance_activity_resolved,activity_resolution_status,activity_anchor_source_row,row_classification
0,2019-20,Transmission lines maintenance,Transmission towers,Number of towers,0's,13204.00,17605.333333,49.797561,1.0,3.0,...,2.8 Maintenance,12,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match,Transmission lines maintenance,submitted,12,meaningful
1,2019-20,Transmission lines maintenance,Transmission tower support structures,Number of towers,0's,69527.00,46351.333333,49.039255,3.0,3.0,...,2.8 Maintenance,13,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match,Transmission lines maintenance,submitted,13,meaningful
2,2019-20,Transmission lines maintenance,Conductors,Route length,km,5086.80,1695.600000,45.342263,3.0,0.0,...,2.8 Maintenance,14,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match,Transmission lines maintenance,submitted,14,meaningful
3,2019-20,Transmission lines maintenance,Transmission cables,Route length,km,8.54,17.080000,26.126850,1.0,1.0,...,2.8 Maintenance,15,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match,Transmission lines maintenance,submitted,15,meaningful
4,2019-20,Substations equipment & property maintenance,Substation switchbays (incl. Reactive plant),Number of switchbays,0's,1133.00,2340.000000,22.911932,1.0,6.0,...,2.8 Maintenance,16,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match,Substations equipment & property maintenance,submitted,16,meaningful


Index(['reporting_period', 'maintenance_activity',
       'maintenance_asset_category', 'measure_asset_quantity', 'source_unit',
       'asset_quantity_at_year_end', 'quantity_inspected_maintained',
       'average_age_of_asset_group', 'inspection_cycle_years',
       'maintenance_cycle_years', 'source_workbook', 'source_sheet',
       'source_row', 'business', 'landing_page_url', 'source_page_url',
       'metadata_match_status', 'maintenance_activity_resolved',
       'activity_resolution_status', 'activity_anchor_source_row',
       'row_classification'],
      dtype='str')

### Verify: all 24 workbooks reconcile to the expected businesses

In [116]:
stage_2a.descriptor_metrics[['source_workbook', 'business']].drop_duplicates()

,source_workbook,business
0,AusNet (T) 2019-20 - Category Analysis - RIN R...,AusNet Transmission
10,AusNet (T) 2020-21 - Category Analysis - RIN R...,AusNet Transmission
20,AusNet (T) 2021-22 - Category Analysis - RIN R...,AusNet Transmission
30,Ausnet Services (ET) 2022-23 - Category Analys...,AusNet Transmission
40,Ausnet Services (ET) 2023-24 - Category Analys...,AusNet Transmission
50,Copy of D14-00150087(V02) - transgrid (t) 2013...,Transgrid
61,Copy of D16 147530(V3) ElectraNet 2015-16 - C...,ElectraNet
71,Copy of D17 153782 ElectraNet 2016-17 - Categ...,ElectraNet
81,DORIS - D18-158142 Transgrid 2017-18 - Categor...,Transgrid
92,DORIS - D18-158267 ElectraNet 2017-18 - Catego...,ElectraNet


### Verify: there are no unexpected unresolved rows;

In [117]:
not(
    'unresolved' in stage_2a.descriptor_metrics['row_classification'].unique()
)

True

### Verify: continued rows inherit the correct parent activity

In [118]:
stage_2a.descriptor_metrics[['business', 'reporting_period', 'maintenance_activity',
       'maintenance_asset_category', 'maintenance_activity_resolved',
       'activity_resolution_status', 'activity_anchor_source_row',
       'row_classification']].query('activity_resolution_status != "submitted" ')

,business,reporting_period,maintenance_activity,maintenance_asset_category,maintenance_activity_resolved,activity_resolution_status,activity_anchor_source_row,row_classification
60,Transgrid,2013-14,NaN,Metering Systems,Other maintenance activity,continued_group_label,21,meaningful
91,Transgrid,2017-18,NaN,Communications,Other maintenance activity,continued_group_label,21,meaningful
173,Powerlink,2019-20,NaN,CORRIDOR MAINTENANCE (NON VEG),Other maintenance activity,continued_group_label,21,meaningful
184,Powerlink,2020-21,NaN,Corridor maintenance (Non vegetation),Other maintenance activity,continued_group_label,21,meaningful
195,Powerlink,2021-22,NaN,Corridor maintenance (non vegetation),Other maintenance activity,continued_group_label,21,meaningful
206,Powerlink,2022-23,NaN,Corridor maintenance (non veg),Other maintenance activity,continued_group_label,22,meaningful
217,Powerlink,2023-24,NaN,Corridor Maintenance (Non-Veg),Other maintenance activity,continued_group_label,22,meaningful
228,Transgrid,2018-19,NaN,Communications,Other maintenance activity,continued_group_label,21,meaningful
239,Transgrid,2019-20,NaN,Communications,Other maintenance activity,continued_group_label,21,meaningful
250,Transgrid,2020-21,NaN,Communications,Other maintenance activity,continued_group_label,21,meaningful


In [119]:
stage_2a.descriptor_metrics[
    stage_2a.descriptor_metrics.row_classification == 'group_header_only'
]

,reporting_period,maintenance_activity,maintenance_asset_category,measure_asset_quantity,source_unit,asset_quantity_at_year_end,quantity_inspected_maintained,average_age_of_asset_group,inspection_cycle_years,maintenance_cycle_years,...,source_sheet,source_row,business,landing_page_url,source_page_url,metadata_match_status,maintenance_activity_resolved,activity_resolution_status,activity_anchor_source_row,row_classification


## Issues, extraction complete

In [120]:
stage_2a.issues

,severity,table_name,source_workbook,source_row,issue_code,message


In [121]:
stage_2a.extraction_complete

True

## Workbook mapping

In [122]:
display(stage_2a.workbook_mapping.head())
display(stage_2a.workbook_mapping.columns)

,source_workbook,extraction_status,run_report_reporting_period,extracted_reporting_period,business_candidate,business,manifest_reporting_period,landing_page_url,source_page_url,metadata_match_status
0,AusNet (T) 2019-20 - Category Analysis - RIN R...,success,2019-20,2019-20,AusNet Transmission,AusNet Transmission,2019-20,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match
1,AusNet (T) 2020-21 - Category Analysis - RIN R...,success,2020-21,2020-21,AusNet Transmission,AusNet Transmission,2020-21,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/authors/ausnet-services...,validated_manifest_match
2,AusNet (T) 2021-22 - Category Analysis - RIN R...,success,2021-22,2021-22,AusNet Transmission,AusNet Transmission,2021-22,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/authors/ausnet-services...,validated_manifest_match
3,Ausnet Services (ET) 2022-23 - Category Analys...,success,2022-23,2022-23,AusNet Transmission,AusNet Transmission,2022-23,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/authors/ausnet-services...,validated_manifest_match
4,Ausnet Services (ET) 2023-24 - Category Analys...,success,2023-24,2023-24,AusNet Transmission,AusNet Transmission,2023-24,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/authors/ausnet-services...,validated_manifest_match


Index(['source_workbook', 'extraction_status', 'run_report_reporting_period',
       'extracted_reporting_period', 'business_candidate', 'business',
       'manifest_reporting_period', 'landing_page_url', 'source_page_url',
       'metadata_match_status'],
      dtype='str')

In [123]:
stage_2a.workbook_mapping.metadata_match_status.unique()

<StringArray>
['validated_manifest_match']
Length: 1, dtype: str

In [124]:
stage_2a.workbook_mapping.extraction_status.unique()

<StringArray>
['success']
Length: 1, dtype: str

In [125]:
all(stage_2a.workbook_mapping.run_report_reporting_period == stage_2a.workbook_mapping.extracted_reporting_period)

True

In [126]:
all(stage_2a.workbook_mapping.business_candidate == stage_2a.workbook_mapping.business)

True

In [127]:
stage_2a.workbook_mapping[['extracted_reporting_period', 'business', 'source_workbook']].\
        sort_values(by=['business', 'extracted_reporting_period'])

,extracted_reporting_period,business,source_workbook
0,2019-20,AusNet Transmission,AusNet (T) 2019-20 - Category Analysis - RIN R...
1,2020-21,AusNet Transmission,AusNet (T) 2020-21 - Category Analysis - RIN R...
2,2021-22,AusNet Transmission,AusNet (T) 2021-22 - Category Analysis - RIN R...
3,2022-23,AusNet Transmission,Ausnet Services (ET) 2022-23 - Category Analys...
4,2023-24,AusNet Transmission,Ausnet Services (ET) 2023-24 - Category Analys...
6,2015-16,ElectraNet,Copy of D16 147530(V3) ElectraNet 2015-16 - C...
7,2016-17,ElectraNet,Copy of D17 153782 ElectraNet 2016-17 - Categ...
9,2017-18,ElectraNet,DORIS - D18-158267 ElectraNet 2017-18 - Catego...
10,2018-19,ElectraNet,ElectraNet 2018-19 - Category Analysis - RIN R...
11,2019-20,ElectraNet,ElectraNet 2019-20 - Category Analysis - RIN R...


# Additional analytics

In [128]:
desc_categs = ['maintenance_activity_resolved', 'maintenance_asset_category']
desc_units = ['measure_asset_quantity', 'source_unit']
desc_nums = ['asset_quantity_at_year_end',
            'quantity_inspected_maintained',
            'average_age_of_asset_group',
            'inspection_cycle_years',
            'maintenance_cycle_years']
cost_categs = ['maintenance_activity_resolved', 'maintenance_asset_subcategory']
cost_units = ['source_currency_unit']
cost_nums = ['routine_maintenance_expenditure',
            'non_routine_maintenance_expenditure']

In [131]:
# business coverage for each reporting period
stage_2a.workbook_mapping.groupby('extracted_reporting_period')['business'].nunique().sort_values()

extracted_reporting_period
2013-14    1
2015-16    1
2016-17    1
2017-18    2
2018-19    2
2019-20    4
2020-21    4
2021-22    4
2022-23    4
2023-24    4
Name: business, dtype: int64

In [130]:
display(stage_2a.descriptor_metrics.head())
display(stage_2a.descriptor_metrics.columns)
display(stage_2a.descriptor_metrics.business.unique())

,reporting_period,maintenance_activity,maintenance_asset_category,measure_asset_quantity,source_unit,asset_quantity_at_year_end,quantity_inspected_maintained,average_age_of_asset_group,inspection_cycle_years,maintenance_cycle_years,...,source_sheet,source_row,business,landing_page_url,source_page_url,metadata_match_status,maintenance_activity_resolved,activity_resolution_status,activity_anchor_source_row,row_classification
0,2019-20,Transmission lines maintenance,Transmission towers,Number of towers,0's,13204.00,17605.333333,49.797561,1.0,3.0,...,2.8 Maintenance,12,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match,Transmission lines maintenance,submitted,12,meaningful
1,2019-20,Transmission lines maintenance,Transmission tower support structures,Number of towers,0's,69527.00,46351.333333,49.039255,3.0,3.0,...,2.8 Maintenance,13,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match,Transmission lines maintenance,submitted,13,meaningful
2,2019-20,Transmission lines maintenance,Conductors,Route length,km,5086.80,1695.600000,45.342263,3.0,0.0,...,2.8 Maintenance,14,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match,Transmission lines maintenance,submitted,14,meaningful
3,2019-20,Transmission lines maintenance,Transmission cables,Route length,km,8.54,17.080000,26.126850,1.0,1.0,...,2.8 Maintenance,15,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match,Transmission lines maintenance,submitted,15,meaningful
4,2019-20,Substations equipment & property maintenance,Substation switchbays (incl. Reactive plant),Number of switchbays,0's,1133.00,2340.000000,22.911932,1.0,6.0,...,2.8 Maintenance,16,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/search?search=AusNet+Tr...,validated_manifest_match,Substations equipment & property maintenance,submitted,16,meaningful


Index(['reporting_period', 'maintenance_activity',
       'maintenance_asset_category', 'measure_asset_quantity', 'source_unit',
       'asset_quantity_at_year_end', 'quantity_inspected_maintained',
       'average_age_of_asset_group', 'inspection_cycle_years',
       'maintenance_cycle_years', 'source_workbook', 'source_sheet',
       'source_row', 'business', 'landing_page_url', 'source_page_url',
       'metadata_match_status', 'maintenance_activity_resolved',
       'activity_resolution_status', 'activity_anchor_source_row',
       'row_classification'],
      dtype='str')

<StringArray>
['AusNet Transmission', 'Transgrid', 'ElectraNet', 'Powerlink']
Length: 4, dtype: str

In [132]:
stage_2a.descriptor_metrics[['business', 'reporting_period',
                       'maintenance_activity_resolved', 'maintenance_asset_category',
                       'measure_asset_quantity', 'source_unit',
                       'asset_quantity_at_year_end', 'quantity_inspected_maintained', 'average_age_of_asset_group', 'inspection_cycle_years', 'maintenance_cycle_years',
                       'activity_resolution_status', 'source_workbook', 'source_row', 'row_classification'
                       ]].query('business=="AusNet Transmission"').sort_values(by=['reporting_period', 'source_row'])

,business,reporting_period,maintenance_activity_resolved,maintenance_asset_category,measure_asset_quantity,source_unit,asset_quantity_at_year_end,quantity_inspected_maintained,average_age_of_asset_group,inspection_cycle_years,maintenance_cycle_years,activity_resolution_status,source_workbook,source_row,row_classification
0,AusNet Transmission,2019-20,Transmission lines maintenance,Transmission towers,Number of towers,0's,13204.000000,17605.333333,49.797561,1.000,3.0,submitted,AusNet (T) 2019-20 - Category Analysis - RIN R...,12,meaningful
1,AusNet Transmission,2019-20,Transmission lines maintenance,Transmission tower support structures,Number of towers,0's,69527.000000,46351.333333,49.039255,3.000,3.0,submitted,AusNet (T) 2019-20 - Category Analysis - RIN R...,13,meaningful
2,AusNet Transmission,2019-20,Transmission lines maintenance,Conductors,Route length,km,5086.800000,1695.600000,45.342263,3.000,0.0,submitted,AusNet (T) 2019-20 - Category Analysis - RIN R...,14,meaningful
3,AusNet Transmission,2019-20,Transmission lines maintenance,Transmission cables,Route length,km,8.540000,17.080000,26.126850,1.000,1.0,submitted,AusNet (T) 2019-20 - Category Analysis - RIN R...,15,meaningful
4,AusNet Transmission,2019-20,Substations equipment & property maintenance,Substation switchbays (incl. Reactive plant),Number of switchbays,0's,1133.000000,2340.000000,22.911932,1.000,6.0,submitted,AusNet (T) 2019-20 - Category Analysis - RIN R...,16,meaningful
5,AusNet Transmission,2019-20,Substations equipment & property maintenance,Substation power transformers,Number of transformers,0's,174.000000,671.000000,25.110256,1.000,4.0,submitted,AusNet (T) 2019-20 - Category Analysis - RIN R...,17,meaningful
6,AusNet Transmission,2019-20,Substations equipment & property maintenance,Substation property,Number of substation properties maintained,0's,47.000000,1660.000000,30.000000,0.100,0.5,submitted,AusNet (T) 2019-20 - Category Analysis - RIN R...,18,meaningful
7,AusNet Transmission,2019-20,SCADA & network control maintenance,SCADA & network control maintenance,Units,0's,19208.548210,9604.274105,18.424151,4.000,4.0,submitted,AusNet (T) 2019-20 - Category Analysis - RIN R...,19,meaningful
8,AusNet Transmission,2019-20,Protection systems maintenance,Protection systems maintenance,Units,0's,6557.000000,4371.333333,15.528810,3.000,3.0,submitted,AusNet (T) 2019-20 - Category Analysis - RIN R...,20,meaningful
9,AusNet Transmission,2019-20,Other maintenance activity,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,submitted,AusNet (T) 2019-20 - Category Analysis - RIN R...,21,meaningful


In [64]:
display(stage_2a.cost_metrics.head())
display(stage_2a.cost_metrics.columns)
display(stage_2a.cost_metrics.business.unique())

,reporting_period,maintenance_activity,maintenance_asset_subcategory,source_currency_unit,routine_maintenance_expenditure,non_routine_maintenance_expenditure,source_workbook,source_sheet,source_row,business,landing_page_url,source_page_url,metadata_match_status,maintenance_activity_resolved,activity_resolution_status,activity_anchor_source_row,row_classification
0,2020-21,Transmission lines maintenance,Transmission towers,$0's,8.056913e+05,9.381098e+05,AusNet (T) 2020-21 - Category Analysis - RIN R...,2.8 Maintenance,50,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/authors/ausnet-services...,validated_manifest_match,Transmission lines maintenance,submitted,50,meaningful
1,2020-21,Transmission lines maintenance,Transmission tower support structures,$0's,2.496602e+05,1.047950e+05,AusNet (T) 2020-21 - Category Analysis - RIN R...,2.8 Maintenance,51,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/authors/ausnet-services...,validated_manifest_match,Transmission lines maintenance,submitted,51,meaningful
2,2020-21,Transmission lines maintenance,Conductors,$0's,1.301578e+06,1.269285e+05,AusNet (T) 2020-21 - Category Analysis - RIN R...,2.8 Maintenance,52,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/authors/ausnet-services...,validated_manifest_match,Transmission lines maintenance,submitted,52,meaningful
3,2020-21,Transmission lines maintenance,Transmission cables,$0's,7.727257e+03,3.535977e+04,AusNet (T) 2020-21 - Category Analysis - RIN R...,2.8 Maintenance,53,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/authors/ausnet-services...,validated_manifest_match,Transmission lines maintenance,submitted,53,meaningful
4,2020-21,Substations equipment & property maintenance,Substation switchbays (incl. Reactive plant),$0's,2.718726e+06,2.782993e+06,AusNet (T) 2020-21 - Category Analysis - RIN R...,2.8 Maintenance,54,AusNet Transmission,https://www.aer.gov.au/documents/ausnet-servic...,https://www.aer.gov.au/authors/ausnet-services...,validated_manifest_match,Substations equipment & property maintenance,submitted,54,meaningful


Index(['reporting_period', 'maintenance_activity',
       'maintenance_asset_subcategory', 'source_currency_unit',
       'routine_maintenance_expenditure',
       'non_routine_maintenance_expenditure', 'source_workbook',
       'source_sheet', 'source_row', 'business', 'landing_page_url',
       'source_page_url', 'metadata_match_status',
       'maintenance_activity_resolved', 'activity_resolution_status',
       'activity_anchor_source_row', 'row_classification'],
      dtype='str')

<StringArray>
['AusNet Transmission', 'Transgrid', 'ElectraNet', 'Powerlink']
Length: 4, dtype: str

In [133]:
stage_2a.cost_metrics[['business', 'reporting_period',
                       'maintenance_activity_resolved', 'maintenance_asset_subcategory',
                       'source_currency_unit',
                       'routine_maintenance_expenditure',
                       'non_routine_maintenance_expenditure', 'activity_resolution_status', 'source_workbook', 'source_row'
                       ]].query('business=="Powerlink"').sort_values(by=['reporting_period', 'source_row'])

,business,reporting_period,maintenance_activity_resolved,maintenance_asset_subcategory,source_currency_unit,routine_maintenance_expenditure,non_routine_maintenance_expenditure,activity_resolution_status,source_workbook,source_row
164,Powerlink,2019-20,Transmission lines maintenance,Transmission towers,$0's,2.376621e+06,1.498310e+07,submitted,Powerlink 2019-20 - Category Analysis - RIN Re...,50
165,Powerlink,2019-20,Transmission lines maintenance,Transmission tower support structures,$0's,8.821804e+04,1.937009e+07,submitted,Powerlink 2019-20 - Category Analysis - RIN Re...,51
166,Powerlink,2019-20,Transmission lines maintenance,Conductors,$0's,0.000000e+00,1.559813e+06,submitted,Powerlink 2019-20 - Category Analysis - RIN Re...,52
167,Powerlink,2019-20,Transmission lines maintenance,Transmission cables,$0's,1.976085e+05,2.157115e+04,submitted,Powerlink 2019-20 - Category Analysis - RIN Re...,53
168,Powerlink,2019-20,Substations equipment & property maintenance,Substation switchbays (incl. Reactive plant),$0's,4.999689e+06,1.630704e+07,submitted,Powerlink 2019-20 - Category Analysis - RIN Re...,54
169,Powerlink,2019-20,Substations equipment & property maintenance,Substation power transformers,$0's,8.383260e+05,2.940998e+06,submitted,Powerlink 2019-20 - Category Analysis - RIN Re...,55
170,Powerlink,2019-20,Substations equipment & property maintenance,Substation property,$0's,1.808051e+06,3.877379e+06,submitted,Powerlink 2019-20 - Category Analysis - RIN Re...,56
171,Powerlink,2019-20,SCADA & network control maintenance,SCADA & network control maintenance,$0's,1.501305e+06,5.836486e+06,submitted,Powerlink 2019-20 - Category Analysis - RIN Re...,57
172,Powerlink,2019-20,Protection systems maintenance,Protection systems maintenance,$0's,9.598954e+05,3.579947e+06,submitted,Powerlink 2019-20 - Category Analysis - RIN Re...,58
173,Powerlink,2019-20,Other maintenance activity,TRANSMISSION TOWERS SUPPORT STRUCTURES,$0's,NaN,NaN,submitted,Powerlink 2019-20 - Category Analysis - RIN Re...,59


## Match categories for normalized expenditure comparison

In [134]:
stage_2a.cost_metrics[['maintenance_activity_resolved', 'maintenance_asset_subcategory']].map(lambda x: x.lower() if isinstance(x, str) else x)

,maintenance_activity_resolved,maintenance_asset_subcategory
0,transmission lines maintenance,transmission towers
1,transmission lines maintenance,transmission tower support structures
2,transmission lines maintenance,conductors
3,transmission lines maintenance,transmission cables
4,substations equipment & property maintenance,substation switchbays (incl. reactive plant)
...,...,...
284,substations equipment & property maintenance,substation property
285,scada & network control maintenance,scada & network control maintenance
286,protection systems maintenance,protection systems maintenance
287,other maintenance activity,metering


In [135]:
stage_2a.descriptor_metrics[['maintenance_activity_resolved', 'maintenance_asset_category']].map(lambda x: x.lower() if isinstance(x, str) else x)

,maintenance_activity_resolved,maintenance_asset_category
0,transmission lines maintenance,transmission towers
1,transmission lines maintenance,transmission tower support structures
2,transmission lines maintenance,conductors
3,transmission lines maintenance,transmission cables
4,substations equipment & property maintenance,substation switchbays (incl. reactive plant)
...,...,...
280,substations equipment & property maintenance,substation property
281,scada & network control maintenance,scada & network control maintenance
282,protection systems maintenance,protection systems maintenance
283,other maintenance activity,metering


# Create conservative category keys

In [136]:
import re
import unicodedata

import pandas as pd


def normalize_category_key(value):
    """Create a conservative comparison key without changing source labels."""

    # Preserve genuine missing values so they are not confused with text.
    if pd.isna(value):
        return pd.NA

    # Normalize Unicode and harmless whitespace or casing differences.
    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", " ", text).strip().casefold()

    # Return a missing value when the submitted text contains no content.
    return text if text else pd.NA


def prepare_category_keys(table, child_column):
    """Add parent-child comparison keys to meaningful Stage 2A rows."""

    # Retain meaningful analytical rows while leaving Stage 2A unchanged.
    prepared = table.loc[
        table["row_classification"].eq("meaningful")
    ].copy()

    # Create conservative keys for the resolved parent and submitted child.
    prepared["maintenance_activity_source_key"] = prepared[
        "maintenance_activity_resolved"
    ].map(normalize_category_key)

    prepared["maintenance_asset_source_key"] = prepared[
        child_column
    ].map(normalize_category_key)

    return prepared


# Prepare separate descriptor and cost working tables.
descriptor_work = prepare_category_keys(
    stage_2a.descriptor_metrics,
    "maintenance_asset_category",
)

cost_work = prepare_category_keys(
    stage_2a.cost_metrics,
    "maintenance_asset_subcategory",
)

# Inspect which source combinations occur in each table

In [137]:
def summarize_category_pairs(table, child_column, table_name):
    """Summarize submitted labels associated with each normalized pair."""

    # Retain the fields needed to review category meaning and coverage.
    review_columns = [
        "maintenance_activity_source_key",
        "maintenance_asset_source_key",
        "maintenance_activity_resolved",
        child_column,
        "business",
        "reporting_period",
    ]

    review = table[review_columns].copy()

    # Summarize the source labels and coverage behind each candidate pair.
    summary = (
        review.groupby(
            [
                "maintenance_activity_source_key",
                "maintenance_asset_source_key",
            ],
            dropna=False,
        )
        .agg(
            submitted_parent_labels=(
                "maintenance_activity_resolved",
                lambda values: sorted(
                    {
                        str(value).strip()
                        for value in values.dropna()
                    }
                ),
            ),
            submitted_child_labels=(
                child_column,
                lambda values: sorted(
                    {
                        str(value).strip()
                        for value in values.dropna()
                    }
                ),
            ),
            businesses=(
                "business",
                lambda values: sorted(set(values.dropna())),
            ),
            reporting_periods=(
                "reporting_period",
                lambda values: sorted(set(values.dropna())),
            ),
            row_count=(child_column, "size"),
        )
        .reset_index()
    )

    # Identify which maintenance table supplied each pair.
    summary.insert(0, "table_name", table_name)

    return summary


# Produce separate inventories before making semantic decisions.
descriptor_inventory = summarize_category_pairs(
    descriptor_work,
    "maintenance_asset_category",
    "descriptor_metrics",
)

cost_inventory = summarize_category_pairs(
    cost_work,
    "maintenance_asset_subcategory",
    "cost_metrics",
)

# Combine the inventories for manual comparison.
category_inventory = pd.concat(
    [descriptor_inventory, cost_inventory],
    ignore_index=True,
)

display(
    category_inventory.sort_values(
        [
            "maintenance_activity_source_key",
            "maintenance_asset_source_key",
            "table_name",
        ]
    )
)

,table_name,maintenance_activity_source_key,maintenance_asset_source_key,submitted_parent_labels,submitted_child_labels,businesses,reporting_periods,row_count
27,cost_metrics,other maintenance activity,bushfire remediation,[Other maintenance activity],[Bushfire Remediation],[Transgrid],"[2019-20, 2020-21, 2021-22, 2022-23]",4
0,descriptor_metrics,other maintenance activity,bushfire remediation,[Other maintenance activity],[Bushfire Remediation],[Transgrid],[2021-22],1
28,cost_metrics,other maintenance activity,communications,[Other maintenance activity],[Communications],[Transgrid],"[2017-18, 2018-19, 2019-20, 2020-21, 2021-22, ...",7
1,descriptor_metrics,other maintenance activity,communications,[Other maintenance activity],[Communications],[Transgrid],"[2017-18, 2018-19, 2019-20, 2020-21, 2021-22, ...",7
29,cost_metrics,other maintenance activity,corridor maintenance (non veg),[Other maintenance activity],"[CORRIDOR MAINTENANCE (NON VEG), Corridor main...",[Powerlink],"[2019-20, 2022-23]",2
2,descriptor_metrics,other maintenance activity,corridor maintenance (non veg),[Other maintenance activity],"[CORRIDOR MAINTENANCE (NON VEG), Corridor main...",[Powerlink],"[2019-20, 2022-23]",2
30,cost_metrics,other maintenance activity,corridor maintenance (non vegetation),[Other maintenance activity],"[Corridor maintenance (Non vegetation), Corrid...",[Powerlink],"[2020-21, 2021-22]",2
3,descriptor_metrics,other maintenance activity,corridor maintenance (non vegetation),[Other maintenance activity],"[Corridor maintenance (Non vegetation), Corrid...",[Powerlink],"[2020-21, 2021-22]",2
31,cost_metrics,other maintenance activity,corridor maintenance (non-veg),[Other maintenance activity],[Corridor Maintenance (Non-Veg)],[Powerlink],[2023-24],1
4,descriptor_metrics,other maintenance activity,corridor maintenance (non-veg),[Other maintenance activity],[Corridor Maintenance (Non-Veg)],[Powerlink],[2023-24],1


That inventory should help distinguish:
- casing or whitespace differences;
- obvious misspellings;
- abbreviations;
- likely aliases needing confirmation;
- genuinely additional categories.

# Define reviewed category decisions

In [138]:
OTHER_ACTIVITY = "Other maintenance activity"

# Record only reviewed semantic equivalences in this mapping.
PAIR_RULES = {
    # Resolve the observed Powerlink spelling error.
    (
        normalize_category_key(OTHER_ACTIVITY),
        normalize_category_key(
            "Tramsission tower support structures"
        ),
    ): (
        OTHER_ACTIVITY,
        "Transmission tower support structures",
    ),

    # Resolve the abbreviated Powerlink label when confirmed equivalent.
    (
        normalize_category_key(OTHER_ACTIVITY),
        normalize_category_key(
            "Transmission Towers Support"
        ),
    ): (
        OTHER_ACTIVITY,
        "Transmission tower support structures",
    ),

    # Consolidate observed non-vegetation corridor wording.
    (
        normalize_category_key(OTHER_ACTIVITY),
        normalize_category_key(
            "Corridor maintenance (Non vegetation)"
        ),
    ): (
        OTHER_ACTIVITY,
        "Corridor maintenance (non-vegetation)",
    ),
    (
        normalize_category_key(OTHER_ACTIVITY),
        normalize_category_key(
            "Corridor maintenance (non veg)"
        ),
    ): (
        OTHER_ACTIVITY,
        "Corridor maintenance (non-vegetation)",
    ),
    (
        normalize_category_key(OTHER_ACTIVITY),
        normalize_category_key(
            "Corridor Maintenance (Non-Veg)"
        ),
    ): (
        OTHER_ACTIVITY,
        "Corridor maintenance (non-vegetation)",
    ),
}

I would leave these out until separately reviewed:
- `ROW Maintenance` versus other right-of-way labels;
- `Telecomms Systems, Telecommunications Systems`, and `Communications`;
- `Metering` versus `Metering Systems`.

They are likely related, but that relationship should ideally be checked against their Basis of Preparation documents.

# apply the mapping without overwriting source columns

In [139]:
def standardize_category_pairs(
    table,
    child_column,
    pair_rules,
):
    """Append reviewed standard labels and stable matching keys."""

    # Copy the table so submitted Stage 2A values remain unchanged.
    standardized = table.copy()

    # Clean presentation whitespace for fallback display labels.
    source_parent = standardized[
        "maintenance_activity_resolved"
    ].map(
        lambda value: (
            re.sub(r"\s+", " ", str(value)).strip()
            if pd.notna(value)
            else pd.NA
        )
    )

    source_child = standardized[child_column].map(
        lambda value: (
            re.sub(r"\s+", " ", str(value)).strip()
            if pd.notna(value)
            else pd.NA
        )
    )

    # Build context-aware source keys from both parent and child.
    source_pair_keys = list(
        zip(
            standardized["maintenance_activity_source_key"],
            standardized["maintenance_asset_source_key"],
        )
    )

    # Look up explicitly reviewed semantic mappings.
    mapped_pairs = pd.Series(
        source_pair_keys,
        index=standardized.index,
    ).map(pair_rules)

    # Use an explicit mapping where one exists; otherwise retain cleaned source labels.
    standardized["maintenance_activity_standard"] = [
        mapped[0] if isinstance(mapped, tuple) else parent
        for mapped, parent in zip(mapped_pairs, source_parent)
    ]

    standardized["maintenance_asset_standard"] = [
        mapped[1] if isinstance(mapped, tuple) else child
        for mapped, child in zip(mapped_pairs, source_child)
    ]

    # Record whether semantic mapping or presentation normalization was applied.
    standardized["category_mapping_status"] = [
        "explicit_alias"
        if isinstance(mapped, tuple)
        else "normalized_source"
        for mapped in mapped_pairs
    ]

    # Create stable keys from the resulting standard labels.
    standardized["maintenance_activity_standard_key"] = standardized[
        "maintenance_activity_standard"
    ].map(normalize_category_key)

    standardized["maintenance_asset_standard_key"] = standardized[
        "maintenance_asset_standard"
    ].map(normalize_category_key)

    return standardized


# Apply the same reviewed semantic rules to both maintenance tables.
descriptor_standardized = standardize_category_pairs(
    descriptor_work,
    "maintenance_asset_category",
    PAIR_RULES,
)

cost_standardized = standardize_category_pairs(
    cost_work,
    "maintenance_asset_subcategory",
    PAIR_RULES,
)

In [140]:
descriptor_standardized

,reporting_period,maintenance_activity,maintenance_asset_category,measure_asset_quantity,source_unit,asset_quantity_at_year_end,quantity_inspected_maintained,average_age_of_asset_group,inspection_cycle_years,maintenance_cycle_years,...,activity_resolution_status,activity_anchor_source_row,row_classification,maintenance_activity_source_key,maintenance_asset_source_key,maintenance_activity_standard,maintenance_asset_standard,category_mapping_status,maintenance_activity_standard_key,maintenance_asset_standard_key
0,2019-20,Transmission lines maintenance,Transmission towers,Number of towers,0's,13204.00,17605.333333,49.797561,1.0,3.0,...,submitted,12,meaningful,transmission lines maintenance,transmission towers,Transmission lines maintenance,Transmission towers,normalized_source,transmission lines maintenance,transmission towers
1,2019-20,Transmission lines maintenance,Transmission tower support structures,Number of towers,0's,69527.00,46351.333333,49.039255,3.0,3.0,...,submitted,13,meaningful,transmission lines maintenance,transmission tower support structures,Transmission lines maintenance,Transmission tower support structures,normalized_source,transmission lines maintenance,transmission tower support structures
2,2019-20,Transmission lines maintenance,Conductors,Route length,km,5086.80,1695.600000,45.342263,3.0,0.0,...,submitted,14,meaningful,transmission lines maintenance,conductors,Transmission lines maintenance,Conductors,normalized_source,transmission lines maintenance,conductors
3,2019-20,Transmission lines maintenance,Transmission cables,Route length,km,8.54,17.080000,26.126850,1.0,1.0,...,submitted,15,meaningful,transmission lines maintenance,transmission cables,Transmission lines maintenance,Transmission cables,normalized_source,transmission lines maintenance,transmission cables
4,2019-20,Substations equipment & property maintenance,Substation switchbays (incl. Reactive plant),Number of switchbays,0's,1133.00,2340.000000,22.911932,1.0,6.0,...,submitted,16,meaningful,substations equipment & property maintenance,substation switchbays (incl. reactive plant),Substations equipment & property maintenance,Substation switchbays (incl. Reactive plant),normalized_source,substations equipment & property maintenance,substation switchbays (incl. reactive plant)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280,2023-24,Substations equipment & property maintenance,Substation property,Number of substation properties maintained,number,96.00,95.000000,39.900000,0.5,0.3,...,submitted,19,meaningful,substations equipment & property maintenance,substation property,Substations equipment & property maintenance,Substation property,normalized_source,substations equipment & property maintenance,substation property
281,2023-24,SCADA & network control maintenance,SCADA & network control maintenance,Number of main control assets,0's,2046.00,196.000000,12.600000,NaN,NaN,...,submitted,20,meaningful,scada & network control maintenance,scada & network control maintenance,SCADA & network control maintenance,SCADA & network control maintenance,normalized_source,scada & network control maintenance,scada & network control maintenance
282,2023-24,Protection systems maintenance,Protection systems maintenance,Number of main protection assets,0's,3286.00,893.000000,15.600000,NaN,6.5,...,submitted,21,meaningful,protection systems maintenance,protection systems maintenance,Protection systems maintenance,Protection systems maintenance,normalized_source,protection systems maintenance,protection systems maintenance
283,2023-24,Other maintenance activity,Metering,Number of main metering assets,0's,838.00,639.000000,10.700000,2.5,5.0,...,submitted,22,meaningful,other maintenance activity,metering,Other maintenance activity,Metering,normalized_source,other maintenance activity,metering


In [141]:
cost_standardized

,reporting_period,maintenance_activity,maintenance_asset_subcategory,source_currency_unit,routine_maintenance_expenditure,non_routine_maintenance_expenditure,source_workbook,source_sheet,source_row,business,...,activity_resolution_status,activity_anchor_source_row,row_classification,maintenance_activity_source_key,maintenance_asset_source_key,maintenance_activity_standard,maintenance_asset_standard,category_mapping_status,maintenance_activity_standard_key,maintenance_asset_standard_key
0,2019-20,Transmission lines maintenance,Transmission towers,$0's,1.090083e+06,8.798923e+05,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,50,AusNet Transmission,...,submitted,50,meaningful,transmission lines maintenance,transmission towers,Transmission lines maintenance,Transmission towers,normalized_source,transmission lines maintenance,transmission towers
1,2019-20,Transmission lines maintenance,Transmission tower support structures,$0's,1.711115e+05,1.089070e+05,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,51,AusNet Transmission,...,submitted,51,meaningful,transmission lines maintenance,transmission tower support structures,Transmission lines maintenance,Transmission tower support structures,normalized_source,transmission lines maintenance,transmission tower support structures
2,2019-20,Transmission lines maintenance,Conductors,$0's,9.506952e+05,4.673437e+05,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,52,AusNet Transmission,...,submitted,52,meaningful,transmission lines maintenance,conductors,Transmission lines maintenance,Conductors,normalized_source,transmission lines maintenance,conductors
3,2019-20,Transmission lines maintenance,Transmission cables,$0's,1.230585e+05,2.483836e+04,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,53,AusNet Transmission,...,submitted,53,meaningful,transmission lines maintenance,transmission cables,Transmission lines maintenance,Transmission cables,normalized_source,transmission lines maintenance,transmission cables
4,2019-20,Substations equipment & property maintenance,Substation switchbays (incl. Reactive plant),$0's,3.323640e+06,1.442763e+06,AusNet (T) 2019-20 - Category Analysis - RIN R...,2.8 Maintenance,54,AusNet Transmission,...,submitted,54,meaningful,substations equipment & property maintenance,substation switchbays (incl. reactive plant),Substations equipment & property maintenance,Substation switchbays (incl. Reactive plant),normalized_source,substations equipment & property maintenance,substation switchbays (incl. reactive plant)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284,2023-24,Substations equipment & property maintenance,Substation property,$,6.745046e+06,4.178675e+06,Transgrid 2023-24 - Category Analysis - RIN Re...,2.8 Maintenance,57,Transgrid,...,submitted,57,meaningful,substations equipment & property maintenance,substation property,Substations equipment & property maintenance,Substation property,normalized_source,substations equipment & property maintenance,substation property
285,2023-24,SCADA & network control maintenance,SCADA & network control maintenance,$,2.443070e+05,7.348910e+05,Transgrid 2023-24 - Category Analysis - RIN Re...,2.8 Maintenance,58,Transgrid,...,submitted,58,meaningful,scada & network control maintenance,scada & network control maintenance,SCADA & network control maintenance,SCADA & network control maintenance,normalized_source,scada & network control maintenance,scada & network control maintenance
286,2023-24,Protection systems maintenance,Protection systems maintenance,$,9.816660e+05,8.727580e+05,Transgrid 2023-24 - Category Analysis - RIN Re...,2.8 Maintenance,59,Transgrid,...,submitted,59,meaningful,protection systems maintenance,protection systems maintenance,Protection systems maintenance,Protection systems maintenance,normalized_source,protection systems maintenance,protection systems maintenance
287,2023-24,Other maintenance activity

# Verify uniqueness before matching
The intended relationship is one descriptor row to one cost row for each standardized business-year category.

In [143]:
# Define the semantic and reporting fields that identify a match.
matching_key = [
    "business",
    "reporting_period",
    "maintenance_activity_standard_key",
    "maintenance_asset_standard_key",
]

# Find category decisions that created duplicate descriptor matches.
duplicate_descriptor_keys = descriptor_standardized.loc[
    descriptor_standardized.duplicated(
        matching_key,
        keep=False,
    )
].sort_values(matching_key)

# Find category decisions that created duplicate cost matches.
duplicate_cost_keys = cost_standardized.loc[
    cost_standardized.duplicated(
        matching_key,
        keep=False,
    )
].sort_values(matching_key)

# Inspect duplicates before allowing a potentially many-to-many join.
display(duplicate_descriptor_keys)
display(duplicate_cost_keys)

# Stop when a mapping decision has made either side ambiguous.
if not duplicate_descriptor_keys.empty:
    raise ValueError(
        "Standardized categories produce duplicate descriptor keys."
    )

if not duplicate_cost_keys.empty:
    raise ValueError(
        "Standardized categories produce duplicate cost keys."
    )

,reporting_period,maintenance_activity,maintenance_asset_category,measure_asset_quantity,source_unit,asset_quantity_at_year_end,quantity_inspected_maintained,average_age_of_asset_group,inspection_cycle_years,maintenance_cycle_years,...,activity_resolution_status,activity_anchor_source_row,row_classification,maintenance_activity_source_key,maintenance_asset_source_key,maintenance_activity_standard,maintenance_asset_standard,category_mapping_status,maintenance_activity_standard_key,maintenance_asset_standard_key


,reporting_period,maintenance_activity,maintenance_asset_subcategory,source_currency_unit,routine_maintenance_expenditure,non_routine_maintenance_expenditure,source_workbook,source_sheet,source_row,business,...,activity_resolution_status,activity_anchor_source_row,row_classification,maintenance_activity_source_key,maintenance_asset_source_key,maintenance_activity_standard,maintenance_asset_standard,category_mapping_status,maintenance_activity_standard_key,maintenance_asset_standard_key


# Match cost rows to descriptor rows

In [144]:
# Select and rename descriptor fields needed by cost-volume analysis.
descriptor_lookup = descriptor_standardized[
    matching_key
    + [
        "maintenance_asset_category",
        "measure_asset_quantity",
        "source_unit",
        "asset_quantity_at_year_end",
        "quantity_inspected_maintained",
        "average_age_of_asset_group",
        "inspection_cycle_years",
        "maintenance_cycle_years",
        "source_workbook",
        "source_row",
    ]
].rename(
    columns={
        "maintenance_asset_category":
            "descriptor_asset_category_source",
        "source_workbook":
            "descriptor_source_workbook",
        "source_row":
            "descriptor_source_row",
    }
)

# Join each cost row to at most one descriptor row.
cost_descriptor_matches = cost_standardized.merge(
    descriptor_lookup,
    on=matching_key,
    how="left",
    validate="one_to_one",
    indicator=True,
)

# Distinguish absent matches from matches without usable denominators.
cost_descriptor_matches[
    "cost_descriptor_match_status"
] = "matched_with_denominator"

cost_descriptor_matches.loc[
    cost_descriptor_matches["_merge"].eq("left_only"),
    "cost_descriptor_match_status",
] = "no_descriptor_match"

cost_descriptor_matches.loc[
    cost_descriptor_matches["_merge"].eq("both")
    & cost_descriptor_matches[
        "asset_quantity_at_year_end"
    ].isna()
    & cost_descriptor_matches[
        "quantity_inspected_maintained"
    ].isna(),
    "cost_descriptor_match_status",
] = "matched_without_denominator"

# Display the relationship inventory before calculating unit-cost measures.
display(
    cost_descriptor_matches[
        [
            "business",
            "reporting_period",
            "maintenance_activity_standard",
            "maintenance_asset_standard",
            "routine_maintenance_expenditure",
            "non_routine_maintenance_expenditure",
            "measure_asset_quantity",
            "source_unit",
            "asset_quantity_at_year_end",
            "quantity_inspected_maintained",
            "cost_descriptor_match_status",
            "maintenance_asset_subcategory",
            "descriptor_asset_category_source",
        ]
    ].sort_values(
        [
            "business",
            "reporting_period",
            "maintenance_activity_standard",
            "maintenance_asset_standard",
        ]
    )
)

,business,reporting_period,maintenance_activity_standard,maintenance_asset_standard,routine_maintenance_expenditure,non_routine_maintenance_expenditure,measure_asset_quantity,source_unit,asset_quantity_at_year_end,quantity_inspected_maintained,cost_descriptor_match_status,maintenance_asset_subcategory,descriptor_asset_category_source
8,AusNet Transmission,2019-20,Protection systems maintenance,Protection systems maintenance,9.289968e+05,7.273859e+05,Units,0's,6557.00000,4371.333333,matched_with_denominator,Protection systems maintenance,Protection systems maintenance
7,AusNet Transmission,2019-20,SCADA & network control maintenance,SCADA & network control maintenance,8.076627e+05,4.352850e+05,Units,0's,19208.54821,9604.274105,matched_with_denominator,SCADA & network control maintenance,SCADA & network control maintenance
5,AusNet Transmission,2019-20,Substations equipment & property maintenance,Substation power transformers,1.895242e+06,2.780589e+06,Number of transformers,0's,174.00000,671.000000,matched_with_denominator,Substation power transformers,Substation power transformers
6,AusNet Transmission,2019-20,Substations equipment & property maintenance,Substation property,1.219192e+06,1.304414e+06,Number of substation properties maintained,0's,47.00000,1660.000000,matched_with_denominator,Substation property,Substation property
4,AusNet Transmission,2019-20,Substations equipment & property maintenance,Substation switchbays (incl. Reactive plant),3.323640e+06,1.442763e+06,Number of switchbays,0's,1133.00000,2340.000000,matched_with_denominator,Substation switchbays (incl. Reactive plant),Substation switchbays (incl. Reactive plant)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
276,Transgrid,2023-24,Substations equipment & property maintenance,Substation switchbays (incl. Reactive plant),1.443408e+06,2.857558e+06,Number of switchbays,number,2350.00000,252.000000,matched_with_denominator,Substation switchbays (incl. Reactive plant),Substation switchbays (incl. Reactive plant)
274,Transgrid,2023-24,Transmission lines maintenance,Conductors,3.032700e+04,6.976210e+05,Route length,km,11230.22000,40.600000,matched_with_denominator,Conductors,Conductors
275,Transgrid,2023-24,Transmission lines maintenance,Transmission cables,6.739990e+05,2.215660e+05,Route length,km,90.78500,90.785000,matched_with_denominator,Transmission cables,Transmission cables
273,Transgrid,2023-24,Transmission lines maintenance,Transmission tower support structures,NaN,NaN,Number of towers,number,NaN,NaN,matched_without_denominator,Transmission tower support structures,Transmission tower support structures
